# 17 — Sync vLLM CrafText rollout

Перед запуском на GPU-сервере подготовьте окружение и локальный snapshot:

```bash
uv sync --extra envs --extra prompts --extra vllm --extra examples
# artifacts/models/qwen25-05b-instruct должен существовать заранее
```

Notebook ничего не скачивает неявно. Если vLLM или snapshot отсутствуют, preflight-ячейка остановится с понятной ошибкой.

Этот notebook доказывает минимальный sync путь: CrafText task → MegaPrompts → Qwen chat template → vLLM batch generation → strict action decode/fallback → batched CrafText step → replay artifacts.

In [ ]:
from __future__ import annotations

from pathlib import Path

from transformers import AutoTokenizer

from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.prompts import MegaPromptRenderer, RenderedPrompt
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.inference import EngineProfile, RequestsLlmBackend, VllmInferenceEngine
from tunix_craftext.rollouts.batched import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.artifacts.replay import save_replay


In [ ]:
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')
SNAPSHOT = ROOT / 'artifacts/models/qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {SNAPSHOT}')

CONFIG_PATH = ROOT / 'configs/mvp/qwen_craftext.yaml'
OUTPUT_DIR = ROOT / 'artifacts/trajectories/vllm-sync-rollout'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('repo:', ROOT)
print('snapshot:', SNAPSHOT)


In [ ]:
config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=config.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
print('instruction_index:', prepared_instruction_index)
print('goal:', prepared_goal)


In [ ]:
base_renderer = MegaPromptRenderer(config.prompt.template)
tokenizer = AutoTokenizer.from_pretrained(str(SNAPSHOT), trust_remote_code=True)

class QwenChatTemplateRenderer:
    def __init__(self, renderer, tokenizer):
        self.renderer = renderer
        self.tokenizer = tokenizer

    def render(self, context):
        rendered = self.renderer.render(context)
        chat_text = self.tokenizer.apply_chat_template(
            [
                {
                    'role': 'system',
                    'content': 'You are a CrafText agent. Return exactly one <action>LABEL</action>.',
                },
                {'role': 'user', 'content': rendered.text},
            ],
            add_generation_prompt=True,
            tokenize=False,
        )
        if not isinstance(chat_text, str) or not chat_text.strip():
            raise ValueError('Qwen chat template did not return text')
        return RenderedPrompt(chat_text, rendered.actions, rendered.template_name)

renderer = QwenChatTemplateRenderer(base_renderer, tokenizer)


In [ ]:
engine_profile = EngineProfile(
    name='qwen25-05b-vllm-sync-rollout',
    backend='vllm-offload',
    model=str(SNAPSHOT),
    tensor_parallel_size=1,
    max_model_len=2048,
    dtype='bfloat16',
    mode='sync',
    metadata={'notebook': '17_sync_vllm_craftext_rollout.ipynb'},
)
engine = VllmInferenceEngine.from_profile(engine_profile)
backend = RequestsLlmBackend(engine)
print('engine:', engine.profile)


In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter,
    renderer,
    backend,
    actions=runtime.actions,
    batch_size=2,
    horizon=4,
    seed=config.run.seed,
    goal=prepared_goal,
    max_new_tokens=8,
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
)

for step, decision in enumerate(rollout.decisions):
    print('step', step)
    print('  actions:', [action.label for action in decision.actions])
    print('  rewards:', decision.transition.reward.tolist())
    print('  fallbacks:', decision.fallback_used.tolist())


In [ ]:
replays = replays_from_batched_rollout(
    rollout,
    config_path=str(CONFIG_PATH.relative_to(ROOT)),
    commit='notebook-sync-vllm',
    backend=f'{engine.profile.backend}:{engine.profile.model}',
)
for env_index, replay in enumerate(replays):
    path = OUTPUT_DIR / f'env-{env_index}.json'
    save_replay(path, replay)
    print('saved', path, 'steps=', len(replay.steps))
